# MiniMind-O Google Colab / Jupyter 全流程交互工程

本项目将 **MiniMind-O**（`~0.1B` 端到端文/音/图多模态 Omni 模型）改造为面向 **Jupyter Notebook / Google Colab** 的交互式工程，无需手动拼接命令行或切换目录，即可在 Notebook 中完成：

1. **环境检测与 Google Drive 持久化**（自动适配 T4 `float16` / L4 & A100 `bfloat16`）
2. **模型与数据集下载**（支持 **HuggingFace** [Colab 推荐] 与 **ModelScope** [国内推荐] 双源）
3. **Parquet 数据集检查与音视频内联预览**
4. **SFT 多阶段训练与 Loss 曲线可视化**（支持 `max_steps` 快速冒烟测试与 `from_resume=1` 断点续训）
5. **PyTorch (`.pth`) ↔ HuggingFace Transformers 格式互转**
6. **Notebook 内多模态对话、浏览器麦克风录音与音色克隆**
7. **一键启动 Gradio Demo (`share=True`) 或实时语音通话 WebUI (`cloudflared` HTTPS/WSS)**

> 💡 **Colab GPU 建议**：免费版 T4 (15GB VRAM) 请使用 `dtype="auto"`（自动切换 `float16`）与 `batch_size <= 16`；L4 / A100 可使用 `bfloat16` 与更大 `batch_size`。

In [ ]:
#@title 1. 环境初始化与依赖安装 (Colab / 本地 Jupyter 通用)
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/jingyaogong/minimind-o.git"  #@param {type:"string"}
BRANCH = ""  #@param {type:"string"} # 留空克隆默认分支，或填入特定分支名
INSTALL_DEPS = True  #@param {type:"boolean"}

def _find_repo_root():
    for cand in [os.getcwd(), os.path.dirname(os.getcwd()), "/content/minimind-o"]:
        if os.path.exists(os.path.join(cand, "notebook_utils.py")) and os.path.exists(os.path.join(cand, "model", "model_omni.py")):
            return os.path.abspath(cand)
    return None

root = _find_repo_root()
if root is None:
    target = "/content/minimind-o" if "google.colab" in sys.modules else os.path.abspath("./minimind-o")
    cmd = ["git", "clone", "--depth", "1"]
    if BRANCH.strip():
        cmd += ["-b", BRANCH.strip()]
    cmd += [REPO_URL, target]
    print("Cloning repository:", " ".join(cmd))
    subprocess.run(cmd, check=True)
    root = target

os.chdir(root)
if root not in sys.path:
    sys.path.insert(0, root)

if INSTALL_DEPS:
    req_file = "requirements_colab.txt" if os.path.exists("requirements_colab.txt") else "requirements.txt"
    print(f"Installing dependencies from {req_file} ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", req_file], check=True)
    if shutil.which("ffmpeg") is None and "google.colab" in sys.modules:
        subprocess.run(["apt-get", "update", "-qq"], check=False)
        subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)

from notebook_utils import check_environment
env_info = check_environment()

In [ ]:
#@title 2. (可选) 挂载 Google Drive 持久化保存权重与断点
USE_GOOGLE_DRIVE = False  #@param {type:"boolean"}
DRIVE_ROOT = "/content/drive/MyDrive/minimind-o"  #@param {type:"string"}

from notebook_utils import setup_persistent_storage

if USE_GOOGLE_DRIVE:
    storage_paths = setup_persistent_storage(drive_root=DRIVE_ROOT, link_out_and_checkpoints=True)
else:
    storage_paths = {
        "save_dir": "./out",
        "checkpoint_dir": "./checkpoints",
        "dataset_dir": "./dataset",
        "hf_dir": "./",
    }
    os.makedirs("./out", exist_ok=True)
    os.makedirs("./checkpoints", exist_ok=True)
    print("Using local workspace directories:", storage_paths)

## 第二步：下载编码器、模型权重与训练数据

- **冻结编码器**（所有推理与训练均需）：`SenseVoiceSmall`（语音）、`siglip2-base-p32-256-ve`（视觉）、`mimi`（音频编解码）、`campplus`（说话人音色克隆）
- **仅体验推理 / WebUI**：选择 `inference_only`
- **从头复现 Mini 训练**：选择 `train_mini` 或 `inference_and_mini_train`

In [ ]:
#@title 3. 下载编码器、权重与数据集
SOURCE = "huggingface"  #@param ["huggingface", "modelscope"]
PRESET = "inference_and_mini_train"  #@param ["inference_only", "train_mini", "inference_and_mini_train", "train_full"]
MODEL_VARIANT = "minimind-3o"  #@param ["minimind-3o", "minimind-3o-moe"]
DOWNLOAD_HF_FORMAT = True  #@param {type:"boolean"}

from notebook_utils import (
    download_encoders,
    download_pytorch_weights,
    download_hf_model,
    download_datasets,
)

download_encoders(source=SOURCE, include_campplus=True)

use_moe = (MODEL_VARIANT == "minimind-3o-moe")
suffix = "_moe" if use_moe else ""

pt_files = []
if "inference" in PRESET:
    pt_files.append(f"sft_omni_768{suffix}.pth")
if "train" in PRESET:
    pt_files.append(f"llm_768{suffix}.pth")

if pt_files:
    download_pytorch_weights(filenames=pt_files, source=SOURCE, save_dir=storage_paths["save_dir"])

if DOWNLOAD_HF_FORMAT and "inference" in PRESET:
    download_hf_model(model_name=MODEL_VARIANT, source=SOURCE, target_dir=os.path.join(".", MODEL_VARIANT))

if "train_mini" in PRESET:
    download_datasets(preset="mini", source=SOURCE, dataset_dir=storage_paths["dataset_dir"])
elif PRESET == "train_full":
    download_datasets(preset="full", source=SOURCE, dataset_dir=storage_paths["dataset_dir"])

## 第三步：数据集结构检查与音频内联试听（可选）

In [ ]:
#@title 4. 预览 Parquet 训练样本（对话文本 + 解码音频）
DATA_FILE = "sft_t2a_mini.parquet"  #@param ["sft_t2a_mini.parquet", "sft_a2a_mini.parquet", "sft_t2a.parquet", "sft_a2a.parquet", "sft_i2t.parquet"]
NUM_SAMPLES = 2  #@param {type:"integer"}

from notebook_utils import inspect_parquet_dataset

parquet_path = os.path.join(storage_paths["dataset_dir"], DATA_FILE)
if not os.path.exists(parquet_path):
    parquet_path = os.path.join("./dataset", DATA_FILE)

if os.path.exists(parquet_path):
    inspect_parquet_dataset(parquet_path, num_samples=NUM_SAMPLES, device=env_info["device"])
else:
    print(f"未找到数据文件: {parquet_path}，请先在上一步选择含训练数据的 PRESET。")

## 第四步：SFT 多阶段训练（Notebook 原生 Python 调用）

参考 `trainer/train.sh` 的推荐阶段：
- **stage1_t2a_mini**：文本到语音对齐（基于 `llm_768.pth`）
- **stage2_a2a_proj_mini**：单独对齐 `audio_proj`（基于上一阶段权重）
- **stage3_a2a_mini**：A2A 全量微调
- **stage4_i2t_proj**：单独对齐 `vision_proj`（需 `sft_i2t.parquet`）

> 💡 初次在 Colab 测试链路可将 `MAX_STEPS` 设为 `30~100`，确认无误后再设为 `0` 跑完整 Epoch。

In [ ]:
#@title 5. 配置并运行 MiniMind-O SFT 训练
RUN_TRAINING = False  #@param {type:"boolean"}
STAGE = "stage1_t2a_mini"  #@param ["stage1_t2a_mini", "stage2_a2a_proj_mini", "stage3_a2a_mini", "stage4_i2t_proj", "custom"]
CUSTOM_DATA_FILE = "sft_t2a_mini.parquet"  #@param ["sft_t2a_mini.parquet", "sft_a2a_mini.parquet", "sft_t2a.parquet", "sft_a2a.parquet", "sft_i2t.parquet"]
MODE = "all"  #@param ["all", "audio_proj", "vision_proj"]
USE_MOE = False  #@param {type:"boolean"}
EPOCHS = 1  #@param {type:"integer"}
BATCH_SIZE = 16  #@param {type:"integer"}
LEARNING_RATE = 5e-4  #@param {type:"number"}
MAX_SEQ_LEN = 512  #@param {type:"integer"}
FROM_WEIGHT = "llm"  #@param {type:"string"}
SAVE_WEIGHT = "sft_zero"  #@param {type:"string"}
FROM_RESUME = False  #@param {type:"boolean"}
MAX_STEPS = 50  #@param {type:"integer"} # 0=跑完整个Epoch；>0=限制步数方便Colab快速验证
MAX_SAMPLES = 0  #@param {type:"integer"} # 0=全量样本；>0=仅取前N条样本
LOG_INTERVAL = 10  #@param {type:"integer"}
SAVE_INTERVAL = 500  #@param {type:"integer"}

from trainer.train_sft_omni import OmniTrainConfig, run_training
from notebook_utils import plot_training_history

stage_presets = {
    "stage1_t2a_mini": dict(data_file="sft_t2a_mini.parquet", mode="all", lr=5e-4, max_seq_len=512, from_weight="llm", save_weight="sft_zero"),
    "stage2_a2a_proj_mini": dict(data_file="sft_a2a_mini.parquet", mode="audio_proj", lr=5e-4, max_seq_len=640, from_weight="sft_zero", save_weight="sft_zero"),
    "stage3_a2a_mini": dict(data_file="sft_a2a_mini.parquet", mode="all", lr=2e-5, max_seq_len=768, from_weight="sft_zero", save_weight="sft_zero"),
    "stage4_i2t_proj": dict(data_file="sft_i2t.parquet", mode="vision_proj", lr=5e-5, max_seq_len=768, from_weight="sft_zero", save_weight="sft_zero"),
}

if RUN_TRAINING:
    preset = stage_presets.get(STAGE, {})
    data_name = preset.get("data_file", CUSTOM_DATA_FILE) if STAGE != "custom" else CUSTOM_DATA_FILE
    data_path = os.path.join(storage_paths["dataset_dir"], data_name)
    if not os.path.exists(data_path):
        data_path = os.path.join("./dataset", data_name)

    cfg = OmniTrainConfig(
        save_dir=storage_paths["save_dir"],
        checkpoint_dir=storage_paths["checkpoint_dir"],
        save_weight=preset.get("save_weight", SAVE_WEIGHT) if STAGE != "custom" else SAVE_WEIGHT,
        from_weight=preset.get("from_weight", FROM_WEIGHT) if STAGE != "custom" else FROM_WEIGHT,
        from_resume=int(FROM_RESUME),
        data_path=data_path,
        epochs=EPOCHS,
        batch_size=min(BATCH_SIZE, env_info["recommended_batch_size"]) if env_info["cuda_available"] else 2,
        learning_rate=preset.get("lr", LEARNING_RATE) if STAGE != "custom" else LEARNING_RATE,
        max_seq_len=preset.get("max_seq_len", MAX_SEQ_LEN) if STAGE != "custom" else MAX_SEQ_LEN,
        mode=preset.get("mode", MODE) if STAGE != "custom" else MODE,
        use_moe=int(USE_MOE),
        dtype=env_info["recommended_dtype"],
        num_workers=env_info["recommended_num_workers"],
        use_compile=0,
        log_interval=LOG_INTERVAL,
        save_interval=SAVE_INTERVAL,
        max_steps=MAX_STEPS,
        max_samples=MAX_SAMPLES,
    )
    print("Training Config:", cfg)
    train_history = run_training(cfg)
    plot_training_history(train_history, title=f"MiniMind-O ({STAGE})")
else:
    print("RUN_TRAINING=False，已跳过训练。如需训练请勾选 RUN_TRAINING=True 后运行。")

## 第五步：将训练得到的 `.pth` 权重转换为 Transformers 格式（可选）

如果打算使用训练后的模型启动 Gradio / 实时 WebUI，需要先导出为 HuggingFace Transformers 目录格式。

In [ ]:
#@title 6. 转换 PyTorch 权重 (.pth) -> HuggingFace 格式
RUN_CONVERT = False  #@param {type:"boolean"}
TORCH_WEIGHT_NAME = "sft_zero"  #@param {type:"string"}
CONVERT_MOE = False  #@param {type:"boolean"}
EXPORT_DIR = "./minimind-3o-custom"  #@param {type:"string"}

from model.model_omni import OmniConfig
from scripts.convert_omni import convert_torch2transformers

if RUN_CONVERT:
    lm_cfg = OmniConfig(hidden_size=768, num_hidden_layers=8, use_moe=CONVERT_MOE)
    moe_suffix = "_moe" if CONVERT_MOE else ""
    torch_path = os.path.join(storage_paths["save_dir"], f"{TORCH_WEIGHT_NAME}_768{moe_suffix}.pth")
    out_dtype = torch.bfloat16 if env_info["bf16_supported"] else torch.float16
    convert_torch2transformers(torch_path, EXPORT_DIR, lm_cfg, dtype=out_dtype)
else:
    print("RUN_CONVERT=False，已跳过格式转换。")

## 第六步：Notebook 内多模态推理、评估与音色克隆

可直接加载 HuggingFace 目录（如 `./minimind-3o`）或原生 `.pth` 权重（`load_from="model"`, `weight="sft_omni"`），并在单元格下方直接听取 24kHz 合成语音。

In [ ]:
#@title 7. 初始化推理会话 (OmniNotebookSession)
LOAD_SOURCE = "hf_dir"  #@param ["hf_dir", "pytorch_pth"]
HF_MODEL_DIR = "./minimind-3o"  #@param {type:"string"}
PT_WEIGHT_PREFIX = "sft_omni"  #@param {type:"string"}
USE_MOE_INFER = False  #@param {type:"boolean"}
OPEN_THINKING = False  #@param {type:"boolean"}

from eval_omni import EvalConfig
from notebook_utils import OmniNotebookSession

eval_cfg = EvalConfig(
    load_from=HF_MODEL_DIR if LOAD_SOURCE == "hf_dir" else "model",
    save_dir=storage_paths["save_dir"],
    weight=PT_WEIGHT_PREFIX,
    use_moe=int(USE_MOE_INFER),
    open_thinking=int(OPEN_THINKING),
    device="cuda" if env_info["cuda_available"] else "cpu",
)
session = OmniNotebookSession(eval_cfg)
print("可用音色:", session.list_voices())

In [ ]:
#@title 8. 文本对话 -> 文本 + 语音输出
PROMPT = "请用简短有趣的话介绍一下你自己。"  #@param {type:"string"}
VOICE_NAME = "default"  #@param ["default", "dylan", "eric", "serena", "uncle_fu", "vivian", "arthur", "chelsie", "cherry", "ethan", "jennifer", "momo", "moon"]
TEMPERATURE = 0.7  #@param {type:"number"}

res = session.chat(text=PROMPT, voice_name=VOICE_NAME, temperature=TEMPERATURE)

In [ ]:
#@title 9. 图像理解 -> 文本 + 语音输出 (支持内置样例或上传图片)
IMAGE_PATH = "./dataset/eval_omni/image-03-panda-holding-hello-sign.jpg"  #@param {type:"string"}
IMAGE_PROMPT = "请描述这张图片里的内容。"  #@param {type:"string"}
UPLOAD_CUSTOM_IMAGE = False  #@param {type:"boolean"}

if UPLOAD_CUSTOM_IMAGE and "google.colab" in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        IMAGE_PATH = list(uploaded.keys())[0]

res_img = session.chat(text=IMAGE_PROMPT, image=IMAGE_PATH)

In [ ]:
#@title 10. 语音输入 -> 文本 + 语音输出 (内置音频 / Colab 麦克风录音)
USE_MIC_RECORDING = False  #@param {type:"boolean"}
MIC_SECONDS = 4  #@param {type:"integer"}
AUDIO_PATH = "./dataset/eval_omni/audio-zh-05_为什么天空是蓝色的.mp3"  #@param {type:"string"}

if USE_MIC_RECORDING:
    from notebook_utils import record_audio_colab
    AUDIO_PATH = record_audio_colab("mic_question.wav", seconds=MIC_SECONDS)

res_audio = session.chat(audio_path=AUDIO_PATH)

In [ ]:
#@title 11. 批量评估套件 (Text / Multi-turn / Audio / Clone / Image / Mix)
EVAL_MODES = "0,4"  #@param {type:"string"} # -1=all, 0=text, 1=multi, 2=audio, 3=clone, 4=image, 5=mix
PROMPT_LANG = 1  #@param {type:"integer"} # 0=英文, 1=中文, 2=中英文
MAX_ITEMS_PER_MODE = 1  #@param {type:"integer"}

eval_results = session.run_eval_suite(mode=EVAL_MODES, prompt_lang=PROMPT_LANG, max_items=MAX_ITEMS_PER_MODE)

## 第七步：启动 WebUI（Gradio / 实时通话模式）

- **Gradio WebUI (`share=True`)**：生成公开 `https://*.gradio.live` 地址，支持文本、图片、音频上传与录音。
- **Realtime Phone-Mode WebUI**：启动 `webui/web_demo.py` 并通过 `cloudflared` 暴露 `https://*.trycloudflare.com`，支持 WSS 实时语音打断（barge-in）与音色克隆！

In [ ]:
#@title 12. 启动 WebUI (二选一)
WEBUI_MODE = "none"  #@param ["none", "gradio", "realtime_flask_cloudflared"]
WEBUI_MODELS_DIR = "./"  #@param {type:"string"} # 自动扫描包含权重文件的模型子目录，如 ./minimind-3o
PORT = 7860  #@param {type:"integer"}

if WEBUI_MODE == "gradio":
    from scripts.web_demo_omni import start_gradio_demo
    start_gradio_demo(load_from=WEBUI_MODELS_DIR, port=PORT, share=True)
elif WEBUI_MODE == "realtime_flask_cloudflared":
    from webui.web_demo import start_realtime_webui
    from notebook_utils import launch_cloudflared_tunnel
    start_realtime_webui(load_from=WEBUI_MODELS_DIR, port=PORT, background=True)
    tunnel_proc, public_url = launch_cloudflared_tunnel(port=PORT)
else:
    print("WEBUI_MODE='none'，如需启动交互界面请选择 gradio 或 realtime_flask_cloudflared。")